In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PostgreSQL Autodownload Example") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2") \
    .getOrCreate()

In [ ]:
import pandas as pd

pd_orders = pd.read_csv("orders.csv").sample(n=1000)

In [ ]:
schema = """
    order_id INT,
    customer_id INT,
    order_date STRING,
    product_name STRING,
    quantity INT,
    price FLOAT,
    total_amount FLOAT,
    status STRING
"""

df_orders = spark.createDataFrame(
    pd_orders,
    schema=schema
)
df_orders.printSchema()

In [ ]:
db_properties = {
    "user": "postgres",
    "password": "mysecretpassword",
    "driver": "org.postgresql.Driver"
}

jdbc_url = "jdbc:postgresql://172.20.131.26:5432/postgres"

df_orders.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .options(**db_properties) \
    .option("dbtable", "orders") \
    .mode("overwrite") \
    .save()

In [ ]:
df = spark.read \
        .format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", "orders") \
        .options(**db_properties) \
        .load()


df.show()

In [ ]:
df_orders.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .options(**db_properties) \
    .option("dbtable", "orders") \
    .option("batchsize", 1000) \
    .mode("overwrite") \
    .save()

In [ ]:
df_orders.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .options(**db_properties) \
    .option("dbtable", "orders") \
    .option("batchsize", 1000) \
    .mode("ignore") \
    .save()

In [ ]:
df_orders.coalesce(1) \
    .write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .options(**db_properties) \
    .option("dbtable", "orders") \
    .option("batchsize", 1000) \
    .mode("overwrite") \
    .save()

In [ ]:
df_orders.repartition(2) \
    .write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .options(**db_properties) \
    .option("dbtable", "orders") \
    .option("batchsize", 1000) \
    .mode("overwrite") \
    .save()

In [ ]:
(
    df_orders.write
        .format("jdbc")
        .option("url", jdbc_url)
        .options(**db_properties)
        .option("dbtable", "orders")
        .mode("overwrite")
        .option(
        "createTableColumnTypes",
        "order_id INTEGER, customer_id INTEGER, product_name VARCHAR(255), quantity INTEGER, price DECIMAL(12, 2), total_amount DOUBLE, status VARCHAR(50)",
        )
        .save()
)

In [ ]:
df_parallel =(
        spark.read
        .format("jdbc")
        .option("url", jdbc_url)
        .options(**db_properties)
        .option("dbtable", "orders")
        .option("partitionColumn", "order_id")
        .option("lowerBound", "1")
        .option("upperBound", "1000")
        .option("numPartitions", "2")
        .load()
)

In [ ]:
df_parallel.count()

In [ ]:
df_parallel.show()

In [ ]:
df_custom = (
    spark.read
    .format("jdbc")
    .option("url", jdbc_url)
    .options(**db_properties)
    .option("query", """
        SELECT order_id, order_date, status
        FROM orders
        WHERE status = 'Processing'
    """)
    .load()
)

df_custom.show(3)

In [ ]:
spark.stop()